In [224]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader, random_split
import torch
import numpy as np
from torch import nn
from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from tqdm import tqdm

In [225]:
df = pd.read_parquet("features.parquet")
df.head()

,features,label
0,"[[0.25699999928474426, 0.2939999997615814, 0.3...",1
1,"[[0.2596735656261444, 0.2984478175640106, 0.32...",1
2,"[[0.26118311285972595, 0.29855456948280334, 0....",1
3,"[[0.2581205368041992, 0.2980491816997528, 0.32...",1
4,"[[0.2573697566986084, 0.297944039106369, 0.319...",1


In [226]:
df.iloc[0]['features']

array([array([0.257     , 0.294     , 0.31999999, 0.33000001, 0.336     ,
              0.249     , 0.28099999, 0.30399999, 0.31900001, 0.229     ,
              0.26100001, 0.287     , 0.30500001, 0.22      , 0.249     ,
              0.27399999, 0.292     , 0.22      , 0.243     , 0.264     ,
              0.28099999, 0.671     , 0.653     , 0.62699997, 0.60699999,
              0.59200001, 0.60000002, 0.57599998, 0.56800002, 0.56599998,
              0.60000002, 0.579     , 0.57099998, 0.56900001, 0.60299999,
              0.583     , 0.57499999, 0.57200003, 0.60699999, 0.588     ,
              0.57999998, 0.57700002, 0.        , 0.        , 0.        ,
              0.        , 0.        , 0.        , 0.        , 0.        ,
              0.        , 0.        , 0.        , 0.        , 0.        ,
              0.        , 0.        , 0.        , 0.        , 0.        ,
              0.        , 0.        , 0.        , 0.        , 0.        ,
              0.        , 0.        , 

In [227]:
max_seq_len = 0
for row in df['features']:
    if len(row) > max_seq_len:
        max_seq_len = len(row)
print(f"Max sequence lenght: {max_seq_len}")

Max sequence lenght: 100


In [228]:
max_seq_len = 100
for row in df['features']:
    tensors = [torch.from_numpy(a).float() for a in row]
    row_tensor = torch.stack(tensors)
    print(row_tensor.shape, end=" ")
    if len(row_tensor) < max_seq_len:
        pad_size = max_seq_len - len(row_tensor)
        padded = torch.cat([
            row_tensor, 
            torch.zeros(pad_size, 84)
        ], dim=0)
    elif len(row_tensor) > max_seq_len:
        padded = row_tensor[:max_seq_len]
    else:
        padded = row_tensor
    print(padded.shape)
    

torch.Size([40, 84]) torch.Size([100, 84])
torch.Size([40, 84]) torch.Size([100, 84])
torch.Size([40, 84]) torch.Size([100, 84])
torch.Size([40, 84]) torch.Size([100, 84])
torch.Size([40, 84]) torch.Size([100, 84])
torch.Size([40, 84]) torch.Size([100, 84])
torch.Size([57, 84]) torch.Size([100, 84])
torch.Size([57, 84]) torch.Size([100, 84])
torch.Size([57, 84]) torch.Size([100, 84])
torch.Size([57, 84]) torch.Size([100, 84])
torch.Size([57, 84]) torch.Size([100, 84])
torch.Size([57, 84]) torch.Size([100, 84])
torch.Size([62, 84]) torch.Size([100, 84])
torch.Size([62, 84]) torch.Size([100, 84])
torch.Size([62, 84]) torch.Size([100, 84])
torch.Size([62, 84]) torch.Size([100, 84])
torch.Size([62, 84]) torch.Size([100, 84])
torch.Size([62, 84]) torch.Size([100, 84])
torch.Size([49, 84]) torch.Size([100, 84])
torch.Size([49, 84]) torch.Size([100, 84])
torch.Size([49, 84]) torch.Size([100, 84])
torch.Size([49, 84]) torch.Size([100, 84])
torch.Size([49, 84]) torch.Size([100, 84])
torch.Size(

In [229]:
class LandmarkDataset(Dataset):
    def __init__(self, dataset_path: str, max_seq_len: int = None):
        self.df = pd.read_parquet(dataset_path)
        if max_seq_len is None:
            max_seq_len = 0
            for row in self.df['features']:
                if len(row) > max_seq_len:
                    max_seq_len = len(row)
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        features = self.df.iloc[idx]['features']
        label = self.df.iloc[idx]['label']
        
        tensors = [torch.from_numpy(a).float() for a in features]
        row_tensor = torch.stack(tensors)
        
        if len(row_tensor) < max_seq_len:
            pad_size = max_seq_len - len(row_tensor)
            padded = torch.cat([
                row_tensor, 
                torch.zeros(pad_size, 84)
            ], dim=0)
        elif len(row_tensor) > max_seq_len:
            padded = row_tensor[:max_seq_len]
        else:
            padded = row_tensor
        
        return padded, label    

In [230]:
dataset = LandmarkDataset("features.parquet")

train_size = int(len(dataset) * 0.8)
val_size = len(dataset) - train_size
    
train_dataset, val_dataset = random_split(
        dataset, [train_size, val_size]
)

train_loader = DataLoader(
        train_dataset,
        batch_size=16,
        shuffle=True,
)
    
val_loader = DataLoader(
        val_dataset,
        batch_size=16
)

In [231]:
def train(
    model,
    epochs,
    optimizer,
    loss_fn
):
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for features, labels in (pbar := tqdm(train_loader, desc=f"Epoch {epoch+1:3d}/{epochs} │ Training", leave=False)):
            optimizer.zero_grad()
            logits = model(features)
            loss = loss_fn(logits, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * features.size(0)
            total_train += labels.size(0)
            
            preds = torch.argmax(logits, dim=1)
            correct_train += (preds == labels).sum().item()
            
            running_loss = train_loss / total_train
            pbar.set_postfix({"loss": f"{running_loss:.4f}"})

        model.eval()
        val_loss = 0.0
        total_val = 0
        correct_val = 0
        
        with torch.inference_mode():
            for features, labels in (pbar := tqdm(val_loader, desc=f"Epoch {epoch+1:3d}/{epochs} │ Validating", leave=False)):
                logits = model(features)
                loss = loss_fn(logits, labels)
                
                val_loss += loss.item() * features.size(0)
                total_val += labels.size(0)
                
                preds = torch.argmax(logits, dim=1)
                correct_val += (preds == labels).sum().item()
                
                running_val_loss = val_loss / total_val
                pbar.set_postfix({"loss": f"{running_val_loss:.4f}"})
        
        avg_train_loss = train_loss / total_train
        avg_val_loss = val_loss / total_val
        
        train_acc = correct_train / total_train
        val_acc = correct_val / total_val
        
        print(
            f"Epoch {epoch+1:3d}/{epochs} │ "
            f"Train Loss: {avg_train_loss:.4f} │ Val Loss: {avg_val_loss:.4f} │ "
            f"Train Acc: {train_acc:.4f} │ Val Acc: {val_acc:.4f}"
        )

In [232]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0)]
        return self.dropout(x)

In [233]:
class GestureTransformer(nn.Module):
    def __init__(
        self,
        num_classes: int = 33,
        d_model: int = 84,
        d_ff: int = 256,
        num_encoders: int = 3,
        nheads: int = 4, 
        dropout: float = 0.3 
    ):
        super().__init__()
        
        # Input projection
        self.input_proj = nn.Linear(84, d_model)
        self.pos_encoding = PositionalEncoding(d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # Transformer encoder only
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nheads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation="relu",
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=num_encoders
        )
        
        # Classifier
        self.fc = nn.Linear(d_model, num_classes)
    
    def forward(self, x):
        batch_size = x.size(0)
        
        # Input projection
        x = self.input_proj(x)  # (batch, seq_len, d_model)
        x = self.pos_encoding(x)
        
        # Add CLS token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)  # (batch, seq_len+1, d_model)
        
        # Transformer encoder
        encoded = self.transformer(x)  # (batch, seq_len+1, d_model)
        
        # Use CLS token for classification
        cls_output = encoded[:, 0, :]  # (batch, d_model)
        
        # Classification
        logits = self.fc(cls_output)  # (batch, num_classes)
        return logits

In [234]:
class GestureRNN(nn.Module):
    def __init__(
        self,
        num_classes: int = 33,
        d_model: int = 84,
        d_hidden: int = 128,
        num_layers: int = 3,
        dropout: float = 0.3
    ):
        super().__init__()  
        
        self.input_proj = nn.Linear(84, d_model)
        
        self.gru = nn.GRU(
            input_size=d_model,
            hidden_size=d_hidden,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.fc = nn.Linear(d_hidden, num_classes)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # Input: (batch_size, seq_len, 84)
        x = self.input_proj(x)  # (batch_size, seq_len, d_model)
        
        # GRU forward
        output, hidden = self.gru(x)  # output: (batch_size, seq_len, d_hidden)
        
        # Use the last output of the sequence
        last_output = output[:, -1, :]  # (batch_size, d_hidden) - last timestep
        
        # Apply dropout and classify
        last_output = self.dropout(last_output)
        logits = self.fc(last_output)  # (batch_size, num_classes)
        
        return logits

In [235]:
class GestureCNN(nn.Module):
    def __init__(
        self,
        num_classes: int = 33,
        d_model: int = 84,
        d_hidden: int = 256,
        dropout: float = 0.3
    ):
        super().__init__()

        self.input_proj = nn.Linear(84, d_model)  
        
        # Convolution layers
        self.convs = nn.Sequential(
            # After input_proj: (batch, seq_len, d_model)  transpose  (batch, d_model, seq_len)
            nn.Conv1d(d_model, d_hidden, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),  # seq_len: 100 → 50
            
            nn.Conv1d(d_hidden, d_hidden * 2, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),  # seq_len: 50 → 25
            
            nn.Conv1d(d_hidden * 2, d_hidden * 4, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)  # (batch, d_hidden*4, 1)
        )   
        
        # Classifier - input size must match last conv layer output channels
        self.fc = nn.Linear(d_hidden * 4, num_classes)   
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x shape: (batch_size, seq_len=100, features=84)
        x = self.input_proj(x)  # (batch_size, 100, d_model)
        x = x.transpose(1, 2)   # (batch_size, d_model, 100)
        
        x = self.convs(x)       # (batch_size, d_hidden*4, 1)
        x = x.squeeze(-1)       # (batch_size, d_hidden*4)
        
        x = self.dropout(x)
        x = self.fc(x)          # (batch_size, num_classes)
        return x

In [236]:
print("Transformer")
model = GestureTransformer(
    num_classes=33,
    d_model=84,
    d_ff=64,
    num_encoders=2,
    nheads=2,
    dropout=0.5
)

optimizer = Adam(model.parameters(), lr=1e-3)
loss_fn = CrossEntropyLoss()

example_input = torch.randn(16, 100, 84)
logits = model(example_input)

print(logits.shape)
train(
    model=model,
    epochs=10,
    optimizer=optimizer,
    loss_fn=loss_fn
)

Transformer
torch.Size([16, 33])


Epoch   1/10 │ Train Loss: 3.5550 │ Val Loss: 3.4973 │ Train Acc: 0.0251 │ Val Acc: 0.0234


Epoch   2/10 │ Train Loss: 3.5260 │ Val Loss: 3.4777 │ Train Acc: 0.0309 │ Val Acc: 0.0365


Epoch   3/10 │ Train Loss: 3.4696 │ Val Loss: 3.4068 │ Train Acc: 0.0452 │ Val Acc: 0.0573


Epoch   4/10 │ Train Loss: 3.4306 │ Val Loss: 3.4750 │ Train Acc: 0.0540 │ Val Acc: 0.0443


...

Epoch   6/10 │ Train Loss: 3.3335 │ Val Loss: 3.4845 │ Train Acc: 0.0684 │ Val Acc: 0.0560


Epoch   7/10 │ Train Loss: 3.2536 │ Val Loss: 3.3817 │ Train Acc: 0.0804 │ Val Acc: 0.0807


Epoch   8/10 │ Train Loss: 3.1649 │ Val Loss: 3.8824 │ Train Acc: 0.0781 │ Val Acc: 0.0690


Epoch   9/10 │ Train Loss: 3.0661 │ Val Loss: 3.1326 │ Train Acc: 0.0986 │ Val Acc: 0.1263


Epoch  10/10 │ Train Loss: 2.9799 │ Val Loss: 3.0514 │ Train Acc: 0.1133 │ Val Acc: 0.1237


In [237]:
print("RNN")
model = GestureRNN(
    num_classes=33,
    d_model=84,
    d_hidden=64,
    num_layers=2,
    dropout=0.3
)
optimizer = Adam(model.parameters(), lr=1e-3)
loss_fn = CrossEntropyLoss()

example_input = torch.randn(16, 100, 84)
logits = model(example_input)

print(logits.shape)
train(
    model=model,
    epochs=10,
    optimizer=optimizer,
    loss_fn=loss_fn
)

RNN
torch.Size([16, 33])


Epoch   1/10 │ Train Loss: 3.4888 │ Val Loss: 3.4840 │ Train Acc: 0.0303 │ Val Acc: 0.0299


Epoch   2/10 │ Train Loss: 3.4753 │ Val Loss: 3.4747 │ Train Acc: 0.0290 │ Val Acc: 0.0339


Epoch   3/10 │ Train Loss: 3.4620 │ Val Loss: 3.4664 │ Train Acc: 0.0361 │ Val Acc: 0.0339


Epoch   4/10 │ Train Loss: 3.4465 │ Val Loss: 3.4522 │ Train Acc: 0.0378 │ Val Acc: 0.0286


...

Epoch   6/10 │ Train Loss: 3.4223 │ Val Loss: 3.4397 │ Train Acc: 0.0534 │ Val Acc: 0.0495


Epoch   7/10 │ Train Loss: 3.4149 │ Val Loss: 3.4286 │ Train Acc: 0.0505 │ Val Acc: 0.0521


Epoch   8/10 │ Train Loss: 3.4031 │ Val Loss: 3.3944 │ Train Acc: 0.0566 │ Val Acc: 0.0534


Epoch   9/10 │ Train Loss: 3.3906 │ Val Loss: 3.3677 │ Train Acc: 0.0586 │ Val Acc: 0.0690


Epoch  10/10 │ Train Loss: 3.3609 │ Val Loss: 3.3521 │ Train Acc: 0.0690 │ Val Acc: 0.0625


In [238]:
print("CNN")
model = GestureCNN(
    num_classes=33,
    d_model=84,
    d_hidden=64,
    dropout=0.3
)
optimizer = Adam(model.parameters(), lr=1e-3)
loss_fn = CrossEntropyLoss()

example_input = torch.randn(16, 100, 84)
logits = model(example_input)

print(logits.shape)
train(
    model=model,
    epochs=10,
    optimizer=optimizer,
    loss_fn=loss_fn
)

CNN
torch.Size([16, 33])


Epoch   1/10 │ Train Loss: 3.4826 │ Val Loss: 3.4513 │ Train Acc: 0.0264 │ Val Acc: 0.0417


Epoch   2/10 │ Train Loss: 3.3798 │ Val Loss: 3.2417 │ Train Acc: 0.0544 │ Val Acc: 0.0547


Epoch   3/10 │ Train Loss: 3.1967 │ Val Loss: 3.1068 │ Train Acc: 0.0716 │ Val Acc: 0.0938


Epoch   4/10 │ Train Loss: 3.1268 │ Val Loss: 2.9742 │ Train Acc: 0.0843 │ Val Acc: 0.1172


...

Epoch   6/10 │ Train Loss: 2.7879 │ Val Loss: 2.6807 │ Train Acc: 0.1348 │ Val Acc: 0.1589


Epoch   7/10 │ Train Loss: 2.6660 │ Val Loss: 2.5076 │ Train Acc: 0.1471 │ Val Acc: 0.2044


Epoch   8/10 │ Train Loss: 2.5562 │ Val Loss: 2.3327 │ Train Acc: 0.1839 │ Val Acc: 0.2513


Epoch   9/10 │ Train Loss: 2.3835 │ Val Loss: 2.0777 │ Train Acc: 0.2214 │ Val Acc: 0.3060


Epoch  10/10 │ Train Loss: 2.1810 │ Val Loss: 1.9020 │ Train Acc: 0.2881 │ Val Acc: 0.3477
